In [106]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

import multiprocessing
import gc

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params
from macromolecules.macromolecule import Macromolecule
from expression.build_me_model import flatten_list
from utils.parameters import human_model as m_model
from core.model import load_pickled_model

In [107]:
mu_val = 1e-9
n_cores = 5
counter = 0

lp_path = '/data2/hratch/human_me/other/test_lp/'
tme0 = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')

In [108]:
def _add_boundary(metabs_, type_ = 'sink', tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        tme_new = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')
    if type(metabs_) != list:
        metabs_ = list(metabs_)
    
    with func.HiddenPrints():
        for m in tqdm(metabs_):
            if isinstance(m, cobra.Metabolite): # object
                tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type =type_)
            else: # string
                tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type =type_)

    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat

def _add_boundary_2(m_ids, mu_val = 1e-9):
    '''Adds a list of m_ids manually as sinks, and solves'''
    
    tme1 = tme0.copy()
    m_ids = [m.id for m in tme1.metabolites]
    
    model_reaction_ids = [r.id for r in tme1.reactions]
    sinks = list()
    for m_id in tqdm(m_ids):
        r = cobra.Reaction('SK_' + m_id)
        r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
        r._lower_bound = -1000
        r._upper_bound = 1000
        if r.id not in model_reaction_ids:
            sinks.append(r)

    print('Add reactions to model')
    tme1.add_reactions(sinks)
    print('solve')
    sln, stat, _ = tme1.solve_lp(mu_val = mu_val)



    reaction_ids = [r.id for r in tme1.reactions if 'biomass' in r.id] 
    res = pd.DataFrame(index = reaction_ids)
    res['reaction_index'] = pd.Series(res.index).apply(lambda x: tme1.reactions.index(x)).tolist()
    res['flux'] = res.reaction_index.apply(lambda x: sln[x])
    return tme1, sln, stat, res


Current status:

In [ ]:
# INFEASIBLE


# FEASIBLE
# all metabolites

# TRY AGAIN (now with non-minimal media model):
# just proteins that are not enzymes - (no one individual protein makes model feasible)
# translation products only
# just ribosomal complexes + HGNC:12458
# increasing and decreasing ribosomal coupling coefficient 1000-fold


# TRY NEXT: 

In [105]:
# # change coupling coefficient of ribosome
# tme1 = tme0.copy()

# for r in tqdm(tme1.reactions):
#     if hasattr(r, 'translation') and r.translation:
#         rbsm = [m for m,v in r.coupled_metabolites.items() if v == 'catalysis'][0]
#         cc = r.metabolites[rbsm]
#         r._metabolites[rbsm] = cc/1000
# sln1, stat1, _ = tme1.solve_lp(mu_val = 1e-9)

In [110]:
# params
mu_val = 1e-9 

tme1 = tme0.copy()

# metabolites only
m_ids = list(set([m.id for m in tme0.metabolites if (not hasattr(m, 'type')) and ('biomass' not in m.id)]))

model_reaction_ids = [r.id for r in tme1.reactions]
sinks = list()
for m_id in tqdm(m_ids):
    r = cobra.Reaction('SK_' + m_id)
    r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
    r._lower_bound = -1000
    r._upper_bound = 1000
    if r.id not in model_reaction_ids:
        sinks.append(r)

print('Add reactions to model')
tme1.add_reactions(sinks)

100%|██████████| 2323/2323 [00:00<00:00, 4144.83it/s]


Add reactions to model


In [111]:
sln1, stat1, _ = tme1.solve_lp(mu_val = 1e-9)

Getting MINOS parameters...
Done in 23.5793 seconds with status 0


In [33]:
sln1, stat1, _ = tme1.solve_lp(mu_val = 0.01)
sln0, stat0, _ = tme0.solve_lp(mu_val = 0.01)

Getting MINOS parameters...
Done in 21.3502 seconds with status 0
Getting MINOS parameters...
Done in 54.475 seconds with status 1


In [34]:
reaction_ids = set([r.id for r in tme0.reactions]).intersection([r.id for r in tme1.reactions])
res = pd.DataFrame(index = reaction_ids)

res['flux_feasible'] = pd.Series(res.index).apply(lambda x: tme1.reactions.index(x)).apply(lambda x: sln1[x]).tolist()
res['flux_infeasible'] = pd.Series(res.index).apply(lambda x: tme0.reactions.index(x)).apply(lambda x: sln0[x]).tolist()
res['flux_diff'] = (res.flux_feasible - res.flux_infeasible).abs()

biomass_ids = [r.id for r in tme0.reactions if 'biomass' in r.id]


ir0 = tme0.infeasible_reactions(mu_val, sln0, stat0, tolerance = 0)
ires = res.loc[ir0.keys(),:]

ires['difference'] = (ires.flux_feasible - ires.flux_infeasible).abs()
ires.sort_values(by = 'difference', ascending = False, inplace = True)

In [35]:
res.loc[biomass_ids, :]

,flux_feasible,flux_infeasible,flux_diff
biomass_dilution,1.000000e-02,1.000000e-02,0.000000e+00
DNA_biomass_to_biomass,1.400000e-04,1.400000e-04,0.000000e+00
carbohydrate_biomass_to_biomass,7.100000e-04,7.100000e-04,0.000000e+00
lipid_biomass_to_biomass,9.700000e-04,9.700000e-04,0.000000e+00
tRNA_biomass_to_biomass,9.559949e-34,8.180721e-03,8.180721e-03
rRNA_biomass_to_biomass,8.611296e-09,0.000000e+00,8.611296e-09
mRNA_biomass_to_biomass,2.074990e-04,1.845735e-23,2.074990e-04
premRNA_biomass_to_biomass,0.000000e+00,0.000000e+00,0.000000e+00
other_RNA_biomass_to_biomass,2.028319e-15,-7.612891e-07,7.612891e-07
protein_biomass_to_biomass,7.972492e-03,4.062455e-08,7.972452e-03


In [36]:
ires.head(10)

,flux_feasible,flux_infeasible,flux_diff,difference
4ABUTtcn,1.000000e+03,-3.099559e-24,1.000000e+03,1.000000e+03
r0921,1.000000e+03,-1.264827e-49,1.000000e+03,1.000000e+03
CLCFTRte_F,5.475989e+01,-4.484685e-05,5.475994e+01,5.475994e+01
FE2DMT1,0.000000e+00,-1.948058e-02,1.948058e-02,1.948058e-02
GALt4_F_0,0.000000e+00,-8.331487e-03,8.331487e-03,8.331487e-03
PGI_F,0.000000e+00,-6.819543e-03,6.819543e-03,6.819543e-03
H2Ot_F_0,0.000000e+00,-8.142618e-04,8.142618e-04,8.142618e-04
NKCCt_F_0,0.000000e+00,-4.492703e-05,4.492703e-05,4.492703e-05
CLS_hs,0.000000e+00,-1.062112e-06,1.062112e-06,1.062112e-06
other_RNA_biomass_to_biomass,2.028319e-15,-7.612891e-07,7.612891e-07,7.612891e-07


[<Metabolite 4abut_n at 0x7fc232688048>,
 <Metabolite 4abut_c at 0x7fc2331ca518>]

In [40]:
tme0.reactions.get_by_id('CLCFTRte_F')

Reaction identifier,CLCFTRte_F
Name,chloride transport by CFTR
Memory address,0x07fc232886160
Stoichiometry,5.681470329875333e-07 HGNC:1884_enzyme_degradation_proxy_pm + 3.09742989626834e5*mu 5.68147032987533e7 HGNC:1884_folded_protein_pm + cl_c --> cl_e 5.681470329875333e-07 + 3.09742989626834e5*mu 5.68147032987533e7 + chloride --> chloride
GPR,HGNC:1884
Lower bound,0
Upper bound,inf


# parallelize above code

In [ ]:
import multiprocessing
import gc
n_cores = 17

tme0 = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')

In [ ]:
def par_sinks(m_id):
    '''Add m_id and solve'''
    
    tme1 = tme0.copy()
    
    model_reaction_ids = [r.id for r in tme1.reactions]
    
    sinks = list()
    r = cobra.Reaction('SK_' + m_id)
    r.add_metabolites({tme1.metabolites.get_by_id(m_id): -1})
    r._lower_bound = -1000
    r._upper_bound = 1000
    if r.id not in model_reaction_ids:
        sinks.append(r)

    tme1.add_reactions(sinks)
    sln, stat, _ = tme1.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_sinks.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')

def add_boundary(m, type_ = 'sink', tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        tme_new = load_pickled_model(lp_path + 'working_version_' + str(counter) + '.pickle')
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type = type_)
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    fn = '/data2/hratch/human_me/other/test_lp/test_' + type_ + '.tab'
    if not os.path.isfile(fn):
        with open(fn, 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open(fn, 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')

In [ ]:
print('Start parallelization')
pool = multiprocessing.Pool(processes = n_cores)
try:
    res = pool.map(par_sinks, m_ids)
    pool.close()
    pool.join()
    gc.collect()
except:
    pool.close()
    pool.join()
    gc.collect()
    raise ValueError('Parallelization failed')    

# print('Start sinks')
# pool = multiprocessing.Pool(processes = n_cores)
# try:
#     res = pool.starmap(add_boundary, zip(metabs_, ['sink']*len(metabs_)))
#     pool.close()
#     pool.join()
#     gc.collect()
# except:
#     pool.close()
#     pool.join()
#     gc.collect()
#     raise ValueError('Parallelization failed')                       

In [58]:
res = pd.read_csv('/data2/hratch/human_me/other/test_lp/test_sinks.tab', sep = '\t')
res.status.unique()